# EN3160 Assignment 1
## Intensity Transformations and Neighborhood Filtering

**Name:** Ranga Rodrigo  
**Index number:** TODO  
**GitHub profile:** TODO  

This notebook contains the implementations, visualizations, and quantitative comparisons for Questions 1-10. Run the setup and asset-preview cells first, then confirm the image mapping in the configuration cell before exporting the report to PDF.

In [ ]:
from pathlib import Path
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from skimage import color, data, transform
from IPython.display import display, Markdown

plt.rcParams.update({'figure.figsize': (10, 5), 'image.cmap': 'gray', 'axes.titlesize': 11})
ROOT = Path.cwd()
IMAGE_FILES = ['contact_lens.tif', 'emma.jpg', 'girl.jpg', 'sigiriya.jpg', 'tom.jpg']

def read_image(filename, grayscale=False):
    path = ROOT / filename
    if grayscale:
        image = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    else:
        image = cv.imread(str(path), cv.IMREAD_COLOR)
        if image is not None:
            image = cv.cvtColor(image, cv.COLOR_BGR2RGB)
    if image is None:
        raise FileNotFoundError(f'Could not read {path}')
    return image

def show_image(ax, image, title='', grayscale=False):
    ax.imshow(image, cmap='gray' if grayscale or image.ndim == 2 else None)
    ax.set_title(title)
    ax.axis('off')

print('Working directory:', ROOT)
print('Assets:', [name for name in IMAGE_FILES if (ROOT / name).exists()])

In [ ]:
fig, axes = plt.subplots(1, len(IMAGE_FILES), figsize=(18, 4))
for ax, filename in zip(axes, IMAGE_FILES):
    try:
        image = read_image(filename)
        show_image(ax, image, filename)
        print(f'{filename}: shape={image.shape}, dtype={image.dtype}')
    except Exception as error:
        ax.set_title(f'{filename}\n{error}')
        ax.axis('off')
plt.tight_layout()
plt.show()

## Configuration and reusable functions

In [ ]:
# Confirm these mappings after viewing the contact sheet above.
Q1_IMAGE = 'contact_lens.tif'
BRAIN_IMAGE = 'contact_lens.tif'
GAMMA_IMAGE = 'emma.jpg'
VIBRANCE_IMAGE = 'girl.jpg'
EQUALIZATION_IMAGE = 'sigiriya.jpg'
FOREGROUND_IMAGE = 'tom.jpg'
SOBEL_IMAGE = 'contact_lens.tif'
ZOOM_LARGE_IMAGE = 'sigiriya.jpg'
ZOOM_SMALL_IMAGE = 'sigiriya.jpg'
GRABCUT_IMAGE = 'tom.jpg'
BILATERAL_IMAGE = 'sigiriya.jpg'

def intensity_transform(im, breakpoints):
    points = np.asarray(breakpoints, dtype=np.float32)
    if points.ndim != 2 or points.shape[1] != 2 or len(points) < 2:
        raise ValueError('breakpoints must have shape (n, 2), with n >= 2')
    points = points[np.argsort(points[:, 0])]
    if np.any(np.diff(points[:, 0]) <= 0):
        raise ValueError('input breakpoints must be strictly increasing')
    output = np.interp(im.astype(np.float32), points[:, 0], points[:, 1])
    return np.clip(output, 0, 255).astype(np.uint8)

def plot_transform(breakpoints, title='Intensity transformation'):
    points = np.asarray(breakpoints)
    plt.figure(figsize=(5, 5))
    plt.plot([0, 255], [0, 255], '--', color='0.65', label='identity')
    plt.plot(points[:, 0], points[:, 1], 'o-', color='tab:red', label='piecewise linear')
    plt.xlim(0, 255); plt.ylim(0, 255)
    plt.xlabel('Input intensity'); plt.ylabel('Output intensity')
    plt.title(title); plt.grid(alpha=.25); plt.legend(); plt.show()

def equalize_uint8(image):
    histogram = np.bincount(image.ravel(), minlength=256)
    cdf = histogram.cumsum()
    nonzero = np.flatnonzero(histogram)
    if len(nonzero) == 0 or cdf[-1] == cdf[nonzero[0]]:
        return image.copy(), histogram, cdf
    cdf_min = cdf[nonzero[0]]
    lut = np.round((cdf - cdf_min) * 255 / (cdf[-1] - cdf_min))
    lut = np.clip(lut, 0, 255).astype(np.uint8)
    return lut[image], histogram, cdf

def foreground_equalize(image, mask):
    result = image.copy()
    foreground = image[mask > 0]
    if foreground.size == 0:
        raise ValueError('The foreground mask is empty')
    histogram = np.bincount(foreground, minlength=256)
    cdf = np.cumsum(histogram)
    nonzero = np.flatnonzero(histogram)
    cdf_min = cdf[nonzero[0]]
    lut = np.clip(np.round((cdf - cdf_min) * 255 / (cdf[-1] - cdf_min)), 0, 255).astype(np.uint8)
    result[mask > 0] = lut[foreground]
    return result, histogram, cdf

def normalized_ssd(first, second):
    a = first.astype(np.float32) / 255
    b = second.astype(np.float32) / 255
    return float(np.mean((a - b) ** 2))

## 1. Piecewise-linear intensity transformation

In [ ]:
q1 = read_image(Q1_IMAGE, grayscale=True)
q1_breakpoints = np.array([[0, 0], [50, 50], [100, 150], [150, 255], [255, 255]])
q1_transformed = intensity_transform(q1, q1_breakpoints)
plot_transform(q1_breakpoints, 'Question 1: piecewise-linear mapping')
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
show_image(axes[0], q1, 'Original image', grayscale=True)
show_image(axes[1], q1_transformed, 'Transformed image', grayscale=True)
plt.show()

## 2. White-matter and gray-matter accentuation

In [ ]:
brain = read_image(BRAIN_IMAGE, grayscale=True)
white_breakpoints = np.array([[0, 0], [70, 20], [120, 255], [255, 255]])
gray_breakpoints = np.array([[0, 0], [70, 0], [120, 220], [170, 255], [255, 255]])
white_accent = intensity_transform(brain, white_breakpoints)
gray_accent = intensity_transform(brain, gray_breakpoints)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
show_image(axes[0, 0], brain, 'Brain proton density', grayscale=True)
show_image(axes[0, 1], white_accent, 'White matter accentuated', grayscale=True)
show_image(axes[1, 0], brain, 'Brain proton density', grayscale=True)
show_image(axes[1, 1], gray_accent, 'Gray matter accentuated', grayscale=True)
for ax, points, title in [(axes[0, 2], white_breakpoints, 'White matter mapping'), (axes[1, 2], gray_breakpoints, 'Gray matter mapping')]:
    ax.plot(points[:, 0], points[:, 1], 'o-'); ax.set(xlim=(0,255), ylim=(0,255), title=title, xlabel='Input', ylabel='Output'); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()

## 3. Gamma correction in the L* plane

In [ ]:
gamma_image = read_image(GAMMA_IMAGE)
gamma = 0.65  # State and justify this value in the discussion.
lab = cv.cvtColor(gamma_image, cv.COLOR_RGB2LAB).astype(np.float32)
l_plane = lab[:, :, 0] / 255
corrected_l = np.clip(255 * (l_plane ** gamma), 0, 255)
lab_corrected = lab.copy(); lab_corrected[:, :, 0] = corrected_l
gamma_corrected = cv.cvtColor(lab_corrected.astype(np.uint8), cv.COLOR_LAB2RGB)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
show_image(axes[0, 0], gamma_image, 'Original')
show_image(axes[0, 1], gamma_corrected, f'Gamma-corrected, gamma={gamma}')
axes[1, 0].hist(gamma_image.ravel(), bins=256, color='0.3'); axes[1, 0].set_title('Original RGB histogram')
axes[1, 1].hist(gamma_corrected.ravel(), bins=256, color='tab:orange'); axes[1, 1].set_title('Corrected RGB histogram')
plt.tight_layout(); plt.show()

## 4. Vibrance enhancement in HSV

In [ ]:
vibrance_image = read_image(VIBRANCE_IMAGE)
hsv = cv.cvtColor(vibrance_image, cv.COLOR_RGB2HSV).astype(np.float32)
a = 0.75
sigma = 70
x = hsv[:, :, 1]
saturation_mapping = np.minimum(x + a * 128 * np.exp(-((x - 128) ** 2) / (2 * sigma ** 2)), 255)
vibrance_hsv = hsv.copy(); vibrance_hsv[:, :, 1] = saturation_mapping
vibrance_result = cv.cvtColor(vibrance_hsv.astype(np.uint8), cv.COLOR_HSV2RGB)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
show_image(axes[0], vibrance_image, 'Original')
show_image(axes[1], vibrance_result, f'Vibrance enhanced, a={a}')
axes[2].plot(np.arange(256), np.arange(256), '--', color='0.6', label='identity')
axes[2].plot(np.arange(256), saturation_mapping[0], alpha=0)
axes[2].plot(np.arange(256), np.minimum(np.arange(256) + a * 128 * np.exp(-((np.arange(256)-128)**2)/(2*sigma**2)), 255), color='tab:green')
axes[2].set(xlim=(0,255), ylim=(0,255), title='Saturation transformation', xlabel='Input saturation', ylabel='Output saturation'); axes[2].grid(alpha=.25)
plt.tight_layout(); plt.show()

## 5. Histogram equalization from first principles

In [ ]:
equalization_input = read_image(EQUALIZATION_IMAGE, grayscale=True)
equalized, hist_before, _ = equalize_uint8(equalization_input)
hist_after = np.bincount(equalized.ravel(), minlength=256)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
show_image(axes[0, 0], equalization_input, 'Original', grayscale=True)
show_image(axes[0, 1], equalized, 'Equalized', grayscale=True)
axes[1, 0].plot(hist_before, color='tab:blue'); axes[1, 0].set_title('Histogram before')
axes[1, 1].plot(hist_after, color='tab:red'); axes[1, 1].set_title('Histogram after')
plt.tight_layout(); plt.show()

## 6. Histogram-equalized foreground

In [ ]:
foreground_rgb = read_image(FOREGROUND_IMAGE)
foreground_hsv = cv.cvtColor(foreground_rgb, cv.COLOR_RGB2HSV)
hue, saturation, value = cv.split(foreground_hsv)
# Threshold the value plane; adjust this threshold after inspecting the three planes.
_, mask = cv.threshold(value, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
# Keep the larger meaningful region as foreground.
if np.mean(mask > 0) > 0.5:
    mask = cv.bitwise_not(mask)
equalized_value, foreground_hist, foreground_cdf = foreground_equalize(value, mask)
background = cv.bitwise_and(foreground_rgb, foreground_rgb, mask=cv.bitwise_not(mask))
foreground_equalized = cv.bitwise_and(foreground_rgb, foreground_rgb, mask=mask)
result_foreground = cv.add(background, cv.cvtColor(cv.merge([hue, saturation, equalized_value]), cv.COLOR_HSV2RGB))
result_foreground = np.where(mask[:, :, None] > 0, result_foreground, foreground_rgb).astype(np.uint8)
fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for ax, plane, title in zip(axes[0, :3], [hue, saturation, value], ['Hue', 'Saturation', 'Value']): show_image(ax, plane, title, grayscale=True)
show_image(axes[0, 3], mask, 'Foreground mask', grayscale=True)
show_image(axes[1, 0], foreground_rgb, 'Original')
show_image(axes[1, 1], foreground_equalized, 'Equalized foreground')
show_image(axes[1, 2], result_foreground, 'Final result')
axes[1, 3].plot(foreground_hist); axes[1, 3].set_title('Foreground histogram before')
plt.tight_layout(); plt.show()

## 7. Sobel filtering: OpenCV, direct convolution, and separable filtering

In [ ]:
sobel_input = read_image(SOBEL_IMAGE, grayscale=True).astype(np.float32)
kernel_x = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=np.float32)
kernel_y = kernel_x.T
opencv_x = cv.filter2D(sobel_input, cv.CV_32F, kernel_x)
direct_x = np.zeros_like(sobel_input)
padded = np.pad(sobel_input, 1, mode='edge')
for row in range(sobel_input.shape[0]):
    for col in range(sobel_input.shape[1]):
        direct_x[row, col] = np.sum(padded[row:row+3, col:col+3] * kernel_x)
separable_x = cv.sepFilter2D(sobel_input, cv.CV_32F, np.array([1,2,1], np.float32), np.array([1,0,-1], np.float32))
def normalize_display(image):
    return cv.normalize(np.abs(image), None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)
fig, axes = plt.subplots(1, 4, figsize=(17, 5))
show_image(axes[0], sobel_input, 'Input', grayscale=True)
show_image(axes[1], normalize_display(opencv_x), 'cv.filter2D', grayscale=True)
show_image(axes[2], normalize_display(direct_x), 'Manual convolution', grayscale=True)
show_image(axes[3], normalize_display(separable_x), 'Separable Sobel', grayscale=True)
print('Direct vs OpenCV max error:', np.max(np.abs(direct_x - opencv_x)))
print('Separable vs OpenCV max error:', np.max(np.abs(separable_x - opencv_x)))
plt.tight_layout(); plt.show()

## 8. Image zooming with nearest-neighbor and bilinear interpolation

In [ ]:
def zoom_image(image, scale, method='nearest'):
    if not (0 < scale <= 10): raise ValueError('scale must be in (0, 10]')
    interpolation = cv.INTER_NEAREST if method == 'nearest' else cv.INTER_LINEAR
    if method not in ('nearest', 'bilinear'): raise ValueError('method must be nearest or bilinear')
    height, width = image.shape[:2]
    return cv.resize(image, (round(width * scale), round(height * scale)), interpolation=interpolation)

zoom_source = read_image(ZOOM_SMALL_IMAGE)
nearest_zoom = zoom_image(zoom_source, 4, 'nearest')
bilinear_zoom = zoom_image(zoom_source, 4, 'bilinear')
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
show_image(axes[0], zoom_source, 'Small input')
show_image(axes[1], nearest_zoom, 'Nearest-neighbor x4')
show_image(axes[2], bilinear_zoom, 'Bilinear x4')
plt.tight_layout(); plt.show()

# For the formal SSD experiment, set ZOOM_LARGE_IMAGE to the matching original.
large = read_image(ZOOM_LARGE_IMAGE)
for method, candidate in [('nearest', nearest_zoom), ('bilinear', bilinear_zoom)]:
    comparable = cv.resize(candidate, (large.shape[1], large.shape[0]), interpolation=cv.INTER_AREA)
    print(method, 'normalized SSD:', normalized_ssd(large, comparable))

## 9. GrabCut segmentation and shallow depth-of-field effect

In [ ]:
grabcut_input = read_image(GRABCUT_IMAGE)
grabcut_bgr = cv.cvtColor(grabcut_input, cv.COLOR_RGB2BGR)
height, width = grabcut_bgr.shape[:2]
rectangle = (int(.03 * width), int(.03 * height), int(.94 * width), int(.94 * height))
labels = np.zeros((height, width), np.uint8)
background_model = np.zeros((1, 65), np.float64)
foreground_model = np.zeros((1, 65), np.float64)
cv.grabCut(grabcut_bgr, labels, rectangle, background_model, foreground_model, 5, cv.GC_INIT_WITH_RECT)
grabcut_mask = np.where((labels == cv.GC_FGD) | (labels == cv.GC_PR_FGD), 255, 0).astype(np.uint8)
foreground = cv.bitwise_and(grabcut_input, grabcut_input, mask=grabcut_mask)
background = cv.bitwise_and(grabcut_input, grabcut_input, mask=cv.bitwise_not(grabcut_mask))
blurred = cv.GaussianBlur(grabcut_input, (0, 0), sigmaX=18)
enhanced = np.where(grabcut_mask[:, :, None] > 0, grabcut_input, blurred).astype(np.uint8)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
show_image(axes[0, 0], grabcut_input, 'Original')
show_image(axes[0, 1], grabcut_mask, 'Final GrabCut mask', grayscale=True)
show_image(axes[0, 2], foreground, 'Foreground')
show_image(axes[1, 0], background, 'Background')
show_image(axes[1, 1], enhanced, 'Blurred-background result')
axes[1, 2].axis('off'); axes[1, 2].text(.05, .8, 'Dark halo explanation:\nThe mask boundary includes pixels\nthat are blended with a blurred\nbackground containing dark pixels.\nThis is a segmentation/feathering\nartifact, not a physical shadow.', fontsize=12, va='top')
plt.tight_layout(); plt.show()

## 10. Bilateral filtering

In [ ]:
def bilateral_filter_gray(image, diameter=5, sigma_space=15, sigma_range=35):
    image = image.astype(np.float32)
    radius = diameter // 2
    padded = np.pad(image, radius, mode='reflect')
    yy, xx = np.mgrid[-radius:radius+1, -radius:radius+1]
    spatial = np.exp(-(xx**2 + yy**2) / (2 * sigma_space**2))
    output = np.zeros_like(image)
    for row in range(image.shape[0]):
        for col in range(image.shape[1]):
            window = padded[row:row+diameter, col:col+diameter]
            range_weights = np.exp(-((window - image[row, col]) ** 2) / (2 * sigma_range**2))
            weights = spatial * range_weights
            output[row, col] = np.sum(weights * window) / np.sum(weights)
    return np.clip(output, 0, 255).astype(np.uint8)

bilateral_rgb = read_image(BILATERAL_IMAGE)
sigma_space, sigma_range = 15, 35
opencv_bilateral = cv.bilateralFilter(bilateral_rgb, 5, sigma_range, sigma_space)
gray_bilateral = cv.cvtColor(bilateral_rgb, cv.COLOR_RGB2GRAY)
manual_bilateral_gray = bilateral_filter_gray(gray_bilateral, 5, sigma_space, sigma_range)
gaussian = cv.GaussianBlur(bilateral_rgb, (5, 5), sigmaX=sigma_space)
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
show_image(axes[0], bilateral_rgb, 'Original')
show_image(axes[1], gaussian, 'Gaussian blur')
show_image(axes[2], opencv_bilateral, 'OpenCV bilateral')
show_image(axes[3], manual_bilateral_gray, 'Manual bilateral (gray)', grayscale=True)
print('Manual/OpenCV normalized SSD:', normalized_ssd(cv.cvtColor(opencv_bilateral, cv.COLOR_RGB2GRAY), manual_bilateral_gray))
plt.tight_layout(); plt.show()

## Discussion and submission checklist

- Replace the TODO fields on the title page with the index number, name, and GitHub profile URL.
- Confirm the image mapping after inspecting the asset preview.
- Record the chosen breakpoints, gamma, vibrance parameter, filter parameters, and normalized SSD values in the written discussion.
- Explain how each transformation changes the relevant histogram or visual feature.
- Commit the notebook regularly to the GitHub repository before exporting.
- Export this notebook directly to PDF and keep the report within the 10-page limit.